In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datasets import load_dataset
import numpy as np
from collections import Counter
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm import tqdm
import os

In [ ]:
ds = load_dataset("cxyzhang/pmc_llmExtraction_v10")
finalized_df = ds["train"].to_pandas()
 

In [ ]:
categories = ['Vitals_Hema', 'Neuro', 'CVS', 'RESP', 'EENT', 'GI', 'GU', 'DERM', 'MSK', 'ENDO', 'LYMPH', "Pregnancy", 'History', 'Lab_Image']

In [ ]:
def process_column_lists(df):
    """
    Converts list values in specified columns to a single string per row,
    but does NOT merge values across different columns.
    """
    for col in categories:
        df[col] = df[col].apply(lambda x: "; ".join(list(set(x))) if isinstance(x,  (list, np.ndarray)) and len(x) > 0 else np.nan)

    return df

In [ ]:
new_df = process_column_lists(finalized_df)

In [ ]:
new_df["topic"] = new_df["pmcid"].astype(str) + ":" + new_df["topic"]

In [ ]:
# def save_aligned_texts(df, categories, embedding_dir="embeddings", text_save_dir="category_texts"):
#     os.makedirs(text_save_dir, exist_ok=True)

#     for category in categories:
#         npz_path = os.path.join(embedding_dir, f"{category}_embeddings.npz")
#         if not os.path.exists(npz_path):
#             print(f"Skipping '{category}': No embedding file found.")
#             continue

#         # Load embeddings
#         npz_data = np.load(npz_path)
#         n_embeddings = npz_data["embeddings"].shape[0]

#         # Get valid texts (same logic as used during embedding generation)
#         category_df = df.dropna(subset=[category]).reset_index(drop=True)
#         valid_texts = [text for text in category_df[category] if isinstance(text, str) and text.strip()]
#         n_texts = len(valid_texts)

#         # Debug check
#         if n_texts != n_embeddings:
#             print(f" Size mismatch for category '{category}':")
#             print(f"    → Valid texts:     {n_texts}")
#             print(f"    → Saved embeddings: {n_embeddings}")
#             print(f"    → Will align on first {min(n_texts, n_embeddings)} entries")

#         # Align and save
#         aligned_texts = valid_texts[:n_embeddings]
#         save_path = os.path.join(text_save_dir, f"{category}_texts.csv")
#         pd.DataFrame({"text": aligned_texts}).to_csv(save_path, index=False)

#         print(f" Saved {len(aligned_texts)} texts for '{category}' to: {save_path}")


In [ ]:
model = "abhinand/MedEmbed-small-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModel.from_pretrained(model)

In [ ]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def generate_embeddings(texts, batch_size=32):
    """
    Generates embeddings for a list of texts using a transformer model.
    Skips empty rows.
    """
     
    embeddings = []
    valid_texts = [text for text in texts if isinstance(text, str) and text.strip()]  # Remove empty values
  
    if not valid_texts:  # If there are no valid texts
        return None

    num_batches = len(valid_texts) // batch_size + (1 if len(valid_texts) % batch_size > 0 else 0)
    
    for i in tqdm(range(0, len(valid_texts), batch_size), desc="Generating embeddings", total=num_batches):
        batch_texts = valid_texts[i:i+batch_size]

        # Tokenization
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {key: val.to(device) for key, val in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # Mean Pooling while ignoring padding tokens
        attention_mask = inputs["attention_mask"].unsqueeze(-1)  # Shape: (batch_size, seq_len, 1)
        masked_embeddings = outputs.last_hidden_state * attention_mask  # Zero out padding tokens
        sum_embeddings = masked_embeddings.sum(dim=1)  # Sum over tokens
        count_tokens = attention_mask.sum(dim=1)  # Count non-padding tokensPe
        mean_pooled = sum_embeddings / count_tokens  # Mean Pooling

        embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(embeddings) if embeddings else None  # Return None if no embeddings

def process_category_embeddings(df, categories, save_dir="embeddings"):
    """
    Processes multiple categories (columns), generates embeddings, and saves
    them along with metadata (topic, age, sex) to separate NPZ files.

    Args:
        df (pd.DataFrame): The input DataFrame.
        categories (list): List of category columns to process.
        save_dir (str): Directory to save the NPZ files.

    Returns:
        None
    """
    import os
    os.makedirs(save_dir, exist_ok=True)  # Ensure the save directory exists

    

    for category in categories:
        if category not in df.columns:
            print(f"Skipping {category}: Column not found in DataFrame.")
            continue

        # Drop rows where the category column is NaN
        category_df = df.dropna(subset=[category]).reset_index(drop=True)

        # Extract relevant texts and metadata
        category_texts = category_df[category].tolist()

        # Generate embeddings
        category_embeddings = generate_embeddings(category_texts)

        # If no embeddings were generated, skip saving
        if category_embeddings is None or category_embeddings.size == 0:
            print(f"Warning: No valid embeddings were generated for {category}. Skipping NPZ saving.")
            continue

        # Keep only rows that had valid embeddings
        valid_indices = category_df.index[:len(category_embeddings)]  # Ensure correct alignment
        filtered_metadata = category_df.loc[valid_indices, ["topic", "age", "sex"]]

        # Convert metadata to NumPy arrays
        topic_data = filtered_metadata["topic"].values
        age_data = filtered_metadata["age"].values
        sex_data = filtered_metadata["sex"].values

        # Define save path
        save_path = os.path.join(save_dir, f"{category}_embeddings.npz")

        # Save to NPZ file
        np.savez_compressed(
            save_path,
            embeddings=category_embeddings,
            topic=topic_data,
            age=age_data,
            sex=sex_data
        )

        print(f"Embeddings and metadata for {category} saved to {save_path}")



# Process and save embeddings for all categories
process_category_embeddings(new_df, categories, save_dir="embeddings")
